# IJMLC English Extension — Resume-safe Confirmatory 5-Seed Runner v10.2

This notebook is designed for **starting fresh or resuming** the English confirmatory experiment.

## Frozen protocol
- Datasets: DailyDialog + MELD
- Backbone: RoBERTa-base
- Models:
  - baseline
  - nrcflat
  - rawgate_zero
  - rawgate_shuffled
  - rawgate
- Seeds: 0, 42, 123, 1234, 2025
- Ontology: ekman_appraisal_ontology_v2.rdf
- Lexical resource: NRC Emotion Lexicon

## Required Kaggle inputs
1. `ijmlc_english_extension_package_v3` (private dataset created from the package ZIP)
2. NRC Emotion Lexicon private dataset
3. Optional: a previously saved `ijmlc_confirmatory_latest.zip` uploaded as a private dataset

The notebook automatically restores completed runs and skips them.

**v10.2 fix:** ultra-robust Kaggle package discovery for renamed/nested/extracted private dataset mounts.

In [ ]:
# ============================================================
# 1. USER CONTROLS
# ============================================================

FULL_SEEDS = [0, 42, 123, 1234, 2025]

CONFIRMATORY_MODELS = [
    "baseline",
    "nrcflat",
    "rawgate_zero",
    "rawgate_shuffled",
    "rawgate",
]

BATCH_SIZE_OVERRIDE = None

REPO_URL = (
    "https://github.com/minkduck/"
    "KG-BERT-Vietnamese-Emotion-Classification.git"
)
REPO_BRANCH = "main"

AUTO_RESTORE_PARTIAL = True

print("FULL_SEEDS           =", FULL_SEEDS)
print("CONFIRMATORY_MODELS  =", CONFIRMATORY_MODELS)
print("BATCH_SIZE_OVERRIDE  =", BATCH_SIZE_OVERRIDE)
print("AUTO_RESTORE_PARTIAL =", AUTO_RESTORE_PARTIAL)

In [ ]:
# ============================================================
# 2. ENVIRONMENT + ULTRA-ROBUST INPUT DISCOVERY — v10.2
# ============================================================

import os
import sys
import json
import csv
import shutil
import zipfile
import subprocess
import urllib.request
import time
from pathlib import Path

os.chdir('/kaggle/working')

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

REPO_DIR = WORK_ROOT / 'KG-BERT-Vietnamese-Emotion-Classification'
DATA_ROOT = WORK_ROOT / 'data' / 'english'
RESULT_ROOT = WORK_ROOT / 'results' / 'english'

DATA_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# NRC discovery
# ------------------------------------------------------------------
nrc_candidates = sorted(
    INPUT_ROOT.rglob('NRC-Emotion-Lexicon-Wordlevel-v0.92.txt')
)

if not nrc_candidates:
    nrc_candidates = sorted(
        p for p in INPUT_ROOT.rglob('*')
        if p.is_file()
        and 'nrc-emotion-lexicon' in p.name.lower()
        and p.suffix.lower() in {'.txt', '.tsv', '.csv'}
    )

if not nrc_candidates:
    raise FileNotFoundError(
        'NRC Emotion Lexicon was not found under /kaggle/input.'
    )

NRC_PATH = nrc_candidates[0]

# ------------------------------------------------------------------
# Resume-safe package-v3 discovery
# ------------------------------------------------------------------
# Kaggle can mount a private dataset in several forms:
#   A) already extracted folder tree,
#   B) intact ZIP,
#   C) archive whose filename/path is changed by Kaggle.
# We therefore use multiple independent discovery strategies.

REQUIRED_REL = [
    Path('scripts/run_experiment.py'),
    Path('scripts/build_english_lexicon.py'),
    Path('src/language_adapters.py'),
    Path('src/ontology_engine.py'),
    Path('src/models/transformer_fusion_models.py'),
    Path('configs/dailydialog_roberta.yaml'),
    Path('configs/meld_roberta.yaml'),
    Path('ontology/ekman_appraisal_ontology_v2.rdf'),
]


def valid_package_root(root: Path) -> bool:
    root = Path(root)
    if not all((root / rel).exists() for rel in REQUIRED_REL):
        return False

    runner = root / 'scripts' / 'run_experiment.py'
    try:
        txt = runner.read_text(encoding='utf-8', errors='replace')
    except Exception:
        return False

    # Accept the known resume-safe marker OR the v3 README marker.
    has_resume_runner = 'SKIP completed model=' in txt
    has_v3_readme = (root / 'README_RESUME_V3.md').exists()

    return has_resume_runner or has_v3_readme


candidate_roots = []

# Strategy 1: anchors from extracted package contents.
anchor_names = {
    'run_experiment.py',
    'README_RESUME_V3.md',
    'PACKAGE_V2_MANIFEST.json',
    'ekman_appraisal_ontology_v2.rdf',
    'dailydialog_roberta.yaml',
}

for anchor in INPUT_ROOT.rglob('*'):
    if not anchor.is_file() or anchor.name not in anchor_names:
        continue

    # Climb up several levels and test each ancestor as package root.
    cur = anchor.parent
    for _ in range(7):
        if cur == INPUT_ROOT or INPUT_ROOT not in cur.parents:
            break
        if valid_package_root(cur):
            candidate_roots.append(cur)
            break
        cur = cur.parent

# Strategy 2: inspect directories whose path looks like the package dataset.
package_dirs = sorted(
    p for p in INPUT_ROOT.rglob('*')
    if p.is_dir()
    and 'ijmlc' in str(p).lower()
    and 'package' in str(p).lower()
    and ('v3' in str(p).lower() or 'extension' in str(p).lower())
)

for base in package_dirs:
    if valid_package_root(base):
        candidate_roots.append(base)

    # A single nested root is common after Kaggle extraction.
    try:
        for child in base.iterdir():
            if child.is_dir() and valid_package_root(child):
                candidate_roots.append(child)
    except Exception:
        pass

# De-duplicate roots while preserving deterministic order.
PACKAGE_ROOTS = []
seen = set()
for p in sorted(candidate_roots, key=lambda x: (len(x.parts), str(x))):
    rp = str(p.resolve())
    if rp not in seen:
        seen.add(rp)
        PACKAGE_ROOTS.append(p)

PACKAGE_ROOT = PACKAGE_ROOTS[0] if PACKAGE_ROOTS else None
PACKAGE_ZIP = None

# Strategy 3: inspect every file in package-looking dataset directories with
# zipfile.is_zipfile(), even when Kaggle removed/changed the .zip extension.
if PACKAGE_ROOT is None:
    archive_candidates = []

    # First limit to package-looking paths.
    for p in INPUT_ROOT.rglob('*'):
        if not p.is_file():
            continue
        low = str(p).lower()
        if 'ijmlc' in low and 'package' in low:
            archive_candidates.append(p)

    # Fallback: all reasonably small files under input (ZIP test is cheap).
    if not archive_candidates:
        archive_candidates = [p for p in INPUT_ROOT.rglob('*') if p.is_file()]

    for p in archive_candidates:
        try:
            if not zipfile.is_zipfile(p):
                continue

            with zipfile.ZipFile(p, 'r') as zf:
                names = zf.namelist()

            has_runner = any(
                n.replace('\\', '/').endswith('scripts/run_experiment.py')
                for n in names
            )
            has_ontology = any(
                n.replace('\\', '/').endswith(
                    'ontology/ekman_appraisal_ontology_v2.rdf'
                )
                for n in names
            )

            if has_runner and has_ontology:
                PACKAGE_ZIP = p
                break
        except Exception:
            continue

# Strategy 4: explicit diagnostics, but only if all discovery failed.
if PACKAGE_ROOT is None and PACKAGE_ZIP is None:
    print('\n=== PACKAGE DISCOVERY DIAGNOSTICS ===')
    print('Top-level /kaggle/input entries:')
    try:
        for p in sorted(INPUT_ROOT.iterdir()):
            print(' ', p)
    except Exception as exc:
        print('Could not list /kaggle/input:', exc)

    print('\nPackage-looking files/directories (first 200):')
    shown = 0
    for p in INPUT_ROOT.rglob('*'):
        low = str(p).lower()
        if 'ijmlc' in low or 'package' in low or 'resume' in low:
            print(' ', p)
            shown += 1
            if shown >= 200:
                break

    raise FileNotFoundError(
        'Resume-safe IJMLC package v3 was not found. '
        'The dataset is visible in Kaggle, but its mounted contents do not '
        'contain the required v3 package files. Re-create the private Kaggle '
        'dataset from ijmlc_english_extension_package_v3.zip without changing '
        'its contents.'
    )

print('=== INPUT DISCOVERY PASSED ===')
print('NRC_PATH     :', NRC_PATH)
print('PACKAGE_ROOT :', PACKAGE_ROOT)
print('PACKAGE_ZIP  :', PACKAGE_ZIP)

if PACKAGE_ROOTS:
    print('All valid package roots:')
    for p in PACKAGE_ROOTS:
        print('  ', p)

# ------------------------------------------------------------------
# GPU check
# ------------------------------------------------------------------
import torch

print('\n=== GPU CHECK ===')
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU is not enabled. Kaggle Settings -> Accelerator -> GPU.'
    )

print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 3. INTERNET CHECK + CLONE + OVERLAY PACKAGE V3 — v10.2
# ============================================================

os.chdir(WORK_ROOT)


def check_url(url, timeout=20):
    try:
        req = urllib.request.Request(
            url,
            headers={'User-Agent': 'Mozilla/5.0 IJMLC-reproducibility'},
        )
        with urllib.request.urlopen(req, timeout=timeout) as response:
            return True, getattr(response, 'status', 200)
    except Exception as exc:
        return False, str(exc)


for url in [
    'https://github.com',
    'https://huggingface.co',
    'https://raw.githubusercontent.com',
]:
    ok, status = check_url(url)
    print(f'{url:40s} -> {status}')
    if not ok:
        raise RuntimeError(
            'Internet access is required. Enable Internet in Kaggle Settings.'
        )

# Leave old repository before deletion.
os.chdir(WORK_ROOT)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone = subprocess.run(
    [
        'git', 'clone',
        '--depth', '1',
        '--branch', REPO_BRANCH,
        REPO_URL,
        str(REPO_DIR),
    ],
    cwd=str(WORK_ROOT),
    text=True,
    capture_output=True,
)

print(clone.stdout)
print(clone.stderr)

if clone.returncode != 0:
    raise RuntimeError('git clone failed.')

# Extract archive package if Cell 2 discovered an archive rather than a folder.
if PACKAGE_ROOT is None:
    overlay_dir = WORK_ROOT / 'ijmlc_overlay_v3'

    if overlay_dir.exists():
        shutil.rmtree(overlay_dir)

    overlay_dir.mkdir(parents=True, exist_ok=True)

    print('Extracting package archive:', PACKAGE_ZIP)

    with zipfile.ZipFile(PACKAGE_ZIP, 'r') as zf:
        zf.extractall(overlay_dir)

    extracted_roots = []

    for runner in overlay_dir.rglob('run_experiment.py'):
        if runner.parent.name != 'scripts':
            continue
        root = runner.parent.parent
        if valid_package_root(root):
            extracted_roots.append(root)

    if not extracted_roots:
        raise RuntimeError(
            'Package archive extracted, but no valid resume-safe v3 package '
            'root was found inside it.'
        )

    PACKAGE_ROOT = sorted(
        extracted_roots,
        key=lambda x: (len(x.parts), str(x)),
    )[0]

print('Overlay source:', PACKAGE_ROOT)

shutil.copytree(
    PACKAGE_ROOT,
    REPO_DIR,
    dirs_exist_ok=True,
)

runner_path = REPO_DIR / 'scripts' / 'run_experiment.py'
runner_text = runner_path.read_text(
    encoding='utf-8',
    errors='replace',
)

if 'SKIP completed model=' not in runner_text:
    raise RuntimeError(
        'The overlaid runner is not the resume-safe v3 runner.'
    )

os.chdir(REPO_DIR)

print('\nCLONE + OVERLAY PASSED.')
print('Resolved PACKAGE_ROOT:', PACKAGE_ROOT)

In [ ]:
# ============================================================
# 4. VERIFY REQUIRED FILES
# ============================================================

required_files = [
    "src/language_adapters.py",
    "src/data_loading.py",
    "src/ontology_engine.py",
    "src/models/transformer_fusion_models.py",
    "scripts/build_english_lexicon.py",
    "scripts/run_experiment.py",
    "configs/dailydialog_roberta.yaml",
    "configs/meld_roberta.yaml",
    "ontology/ekman_appraisal_ontology_v2.rdf",
]

all_ok = True

for rel in required_files:
    p = REPO_DIR / rel
    ok = p.exists()
    print(f"{'OK' if ok else 'MISSING':8s} {rel}")
    all_ok &= ok

if not all_ok:
    raise RuntimeError("Required package files are incomplete.")

print("\nRESUME-SAFE PACKAGE V3 VERIFIED.")

In [ ]:
# ============================================================
# 5. KAGGLE-SAFE DEPENDENCY CHECK
# ============================================================

import importlib

CORE_MODULES = [
    "torch",
    "transformers",
    "numpy",
    "pandas",
    "sklearn",
    "scipy",
    "pyarrow",
]

LIGHT_MODULE_TO_PIP = {
    "rdflib": "rdflib",
    "yaml": "PyYAML",
    "unidecode": "Unidecode",
    "tqdm": "tqdm",
    "accelerate": "accelerate",
    "packaging": "packaging",
}

missing_core = []

for module_name in CORE_MODULES:
    try:
        importlib.import_module(module_name)
    except Exception:
        missing_core.append(module_name)

if missing_core:
    raise RuntimeError(
        "Missing Kaggle core modules: " + ", ".join(missing_core)
    )

for module_name, pip_name in LIGHT_MODULE_TO_PIP.items():
    try:
        importlib.import_module(module_name)
    except Exception:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-deps",
                pip_name,
            ],
            check=True,
        )

if BATCH_SIZE_OVERRIDE is not None:
    import yaml

    for config_name in [
        "configs/dailydialog_roberta.yaml",
        "configs/meld_roberta.yaml",
    ]:
        p = REPO_DIR / config_name
        cfg = yaml.safe_load(p.read_text(encoding="utf-8"))
        cfg["training"]["batch_size"] = int(BATCH_SIZE_OVERRIDE)
        p.write_text(
            yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True),
            encoding="utf-8",
        )

import pandas as pd
import numpy as np
import scipy
import sklearn
import pyarrow
import rdflib
import accelerate

print("torch       :", torch.__version__)
print("transformers:", __import__("transformers").__version__)
print("pandas      :", pd.__version__)
print("sklearn     :", sklearn.__version__)
print("scipy       :", scipy.__version__)
print("pyarrow     :", pyarrow.__version__)
print("rdflib      :", rdflib.__version__)
print("accelerate  :", accelerate.__version__)
print("CUDA        :", torch.cuda.is_available())

print("\nDEPENDENCY CHECK PASSED.")

In [ ]:
# ============================================================
# 6. PREPARE DAILYDIALOG
# ============================================================

import pandas as pd
import numpy as np

DD_DIR = DATA_ROOT / "dailydialog"
DD_DIR.mkdir(parents=True, exist_ok=True)

DD_PARQUET_DIR = DD_DIR / "source_parquet"
DD_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

DD_CSV_DIR = DD_DIR / "normalized_csv"
DD_CSV_DIR.mkdir(parents=True, exist_ok=True)

HF_BASE = (
    "https://huggingface.co/datasets/"
    "DeepPavlov/daily_dialog/resolve/main/data"
)

DD_REMOTE_FILES = {
    "train": "train-00000-of-00001.parquet",
    "validation": "validation-00000-of-00001.parquet",
    "test": "test-00000-of-00001.parquet",
}

def download_public_file(url, destination, retries=4):
    destination = Path(destination)

    if destination.exists() and destination.stat().st_size > 0:
        return

    last_error = None

    for attempt in range(1, retries + 1):
        tmp = destination.with_suffix(destination.suffix + ".part")

        try:
            req = urllib.request.Request(
                url,
                headers={"User-Agent": "Mozilla/5.0 IJMLC-reproducibility"}
            )

            with urllib.request.urlopen(req, timeout=300) as response, \
                    open(tmp, "wb") as fout:
                shutil.copyfileobj(response, fout)

            if tmp.stat().st_size == 0:
                raise RuntimeError("Downloaded file is empty.")

            tmp.replace(destination)
            return

        except Exception as exc:
            last_error = exc
            if tmp.exists():
                tmp.unlink()
            if attempt < retries:
                time.sleep(3 * attempt)

    raise RuntimeError(
        f"Failed to download {url}: {last_error}"
    )

local_parquets = {}

for split, filename in DD_REMOTE_FILES.items():
    p = DD_PARQUET_DIR / filename
    download_public_file(f"{HF_BASE}/{filename}", p)
    local_parquets[split] = p

LABEL_MAP = {
    0: "Neutral",
    1: "Anger",
    2: "Disgust",
    3: "Fear",
    4: "Happiness",
    5: "Sadness",
    6: "Surprise",
}

def last_dialog_item(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple)):
        return "" if not value else str(value[-1]).strip()
    return str(value).strip()

def convert_split(path, split):
    df = pd.read_parquet(path)

    if "emotion_label" in df.columns and "dialog" in df.columns:
        out = pd.DataFrame({
            "Sentence": df["dialog"].map(last_dialog_item),
            "Emotion": df["emotion_label"].map(LABEL_MAP),
        })

    elif "emotion" in df.columns and "dialog" in df.columns:
        rows = []

        for idx, row in df.iterrows():
            dialogs = row["dialog"]
            emotions = row["emotion"]

            if isinstance(dialogs, np.ndarray):
                dialogs = dialogs.tolist()
            if isinstance(emotions, np.ndarray):
                emotions = emotions.tolist()

            if len(dialogs) != len(emotions):
                raise ValueError(
                    f"{split} row {idx}: "
                    f"{len(dialogs)} utterances != {len(emotions)} labels"
                )

            for text, emo in zip(dialogs, emotions):
                rows.append({
                    "Sentence": str(text).strip(),
                    "Emotion": LABEL_MAP[int(emo)],
                })

        out = pd.DataFrame(rows)

    else:
        raise ValueError(
            f"Unsupported DailyDialog schema: {list(df.columns)}"
        )

    if out["Emotion"].isna().any():
        raise ValueError(f"{split}: invalid emotion labels.")

    if out["Sentence"].astype(str).str.strip().eq("").any():
        raise ValueError(f"{split}: empty utterance detected.")

    return out.reset_index(drop=True)

dd_train_raw = convert_split(local_parquets["train"], "train")
dd_dev_raw = convert_split(local_parquets["validation"], "validation")
dd_test_raw = convert_split(local_parquets["test"], "test")

expected = {
    "train": 87170,
    "validation": 8069,
    "test": 7740,
}

actual = {
    "train": len(dd_train_raw),
    "validation": len(dd_dev_raw),
    "test": len(dd_test_raw),
}

for split in expected:
    print(
        f"{split:10s}: {actual[split]:6d} "
        f"(expected {expected[split]:6d})"
    )
    assert actual[split] == expected[split]

dd_train_raw.to_csv(DD_CSV_DIR / "train.csv", index=False)
dd_dev_raw.to_csv(DD_CSV_DIR / "dev.csv", index=False)
dd_test_raw.to_csv(DD_CSV_DIR / "test.csv", index=False)

print("\nDailyDialog preparation PASSED.")

In [ ]:
# ============================================================
# 7. PREPARE MELD
# ============================================================

MELD_DIR = DATA_ROOT / "meld"
MELD_DIR.mkdir(parents=True, exist_ok=True)

MELD_BASE = (
    "https://raw.githubusercontent.com/"
    "declare-lab/MELD/master/data/MELD"
)

meld_names = [
    "train_sent_emo.csv",
    "dev_sent_emo.csv",
    "test_sent_emo.csv",
]

def download_meld(url, destination):
    destination = Path(destination)

    if destination.exists() and destination.stat().st_size > 0:
        return

    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 IJMLC-reproducibility"}
    )

    with urllib.request.urlopen(req, timeout=120) as response, \
            open(destination, "wb") as fout:
        shutil.copyfileobj(response, fout)

    if destination.stat().st_size == 0:
        raise RuntimeError(f"Empty file: {destination}")

for name in meld_names:
    download_meld(
        f"{MELD_BASE}/{name}",
        MELD_DIR / name,
    )

for name in meld_names:
    frame = pd.read_csv(MELD_DIR / name, nrows=5)
    missing = {"Utterance", "Emotion"} - set(frame.columns)
    if missing:
        raise ValueError(
            f"{name} missing columns: {sorted(missing)}"
        )

print("\nMELD download OK.")

In [ ]:
# ============================================================
# 8. BUILD NRC + ONTOLOGY-V2 ENGLISH LEXICAL LAYER
# ============================================================

os.chdir(REPO_DIR)

build_cmd = [
    sys.executable,
    "scripts/build_english_lexicon.py",
    "--nrc", str(NRC_PATH),
    "--ontology", "ontology/ekman_appraisal_ontology_v2.rdf",
    "--output", "ontology/english_nrc_ontology_v2_structured.json",
    "--flat-output", "ontology/english_nrc_flat7.json",
    "--summary-output",
    "ontology/english_nrc_ontology_v2_build_summary.json",
    "--confidence", "0.50",
]

subprocess.run(build_cmd, cwd=REPO_DIR, check=True)

summary_path = (
    REPO_DIR
    / "ontology"
    / "english_nrc_ontology_v2_build_summary.json"
)

build_summary = json.loads(
    summary_path.read_text(encoding="utf-8")
)

print("Structured keys:", build_summary["structured_keys"])
print(
    "Appraisal-enriched keys:",
    build_summary["appraisal_enriched_keys"],
)

print("\nNRC + ontology-v2 lexical layer PASSED.")

In [ ]:
# ============================================================
# 9. DATA + 24-D ONTOLOGY SANITY CHECK
# ============================================================

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.data_loading import load_dailydialog, load_meld
from src.language_adapters import EnglishAdapter
from src.ontology_engine import OntologyEngine

dd_train, dd_dev, dd_test = load_dailydialog(str(DD_DIR))
me_train, me_dev, me_test = load_meld(str(MELD_DIR))

engine = OntologyEngine(
    rdf_path=str(
        REPO_DIR / "ontology" / "ekman_appraisal_ontology_v2.rdf"
    ),
    default_conf=0.50,
    manual_boost=None,
    negation_attenuation=0.50,
    alpha_similarity=0.20,
    verbose=False,
    language_adapter=EnglishAdapter(),
    external_lexicon_path=str(
        REPO_DIR
        / "ontology"
        / "english_nrc_ontology_v2_structured.json"
    ),
)

vec, tokens, trace = engine.getvector(
    "I am angry and sad about this.",
    debug=True,
)

print("Vector shape:", vec.shape)
print("Appraisal block:", vec[7:16])

assert vec.shape == (24,)
assert np.any(np.abs(vec[7:16]) > 1e-8)

print("\nAppraisal dimensions are active: PASS")
print("SANITY CHECKS PASSED.")

In [ ]:
# ============================================================
# 10. RESTORE PREVIOUS PARTIAL RESULTS
# ============================================================

def restore_partial_results():
    if not AUTO_RESTORE_PARTIAL:
        print("Restore disabled.")
        return

    # ZIP may be intact.
    zip_candidates = sorted(
        p for p in INPUT_ROOT.rglob("*.zip")
        if (
            "confirmatory_latest" in p.name.lower()
            or "confirmatory_partial" in p.name.lower()
            or "confirmatory_5seed" in p.name.lower()
        )
        and "package" not in p.name.lower()
    )

    restore_root = None

    if zip_candidates:
        zpath = zip_candidates[-1]
        tmp = WORK_ROOT / "_restore_partial"

        if tmp.exists():
            shutil.rmtree(tmp)

        tmp.mkdir(parents=True, exist_ok=True)

        print("Restoring ZIP:", zpath)

        with zipfile.ZipFile(zpath, "r") as z:
            z.extractall(tmp)

        restore_root = tmp

    else:
        # Or Kaggle may have auto-extracted the ZIP dataset.
        manifests = sorted(
            INPUT_ROOT.rglob("resume_manifest.csv")
        )

        if manifests:
            restore_root = manifests[-1].parent
            print("Restoring extracted bundle:", restore_root)

    if restore_root is None:
        print("No previous partial-results bundle attached.")
        return

    candidates = []

    direct = restore_root / "results" / "english"

    if direct.exists():
        candidates.append(direct)

    candidates.extend(
        p for p in restore_root.rglob("english")
        if p.is_dir() and p.parent.name == "results"
    )

    if not candidates:
        raise RuntimeError(
            "Partial bundle found but results/english is missing."
        )

    src = candidates[0]

    shutil.copytree(
        src,
        RESULT_ROOT,
        dirs_exist_ok=True,
    )

    metrics = list(
        p for p in RESULT_ROOT.rglob("metrics.json")
        if "confirmatory_5seed" in str(p)
    )

    print("Restored metrics files:", len(metrics))
    print("PARTIAL RESULTS RESTORE PASSED.")

restore_partial_results()

In [ ]:
# ============================================================
# 11. RUN CONFIRMATORY 5-SEED EXPERIMENT — RESUME-SAFE
# ============================================================

def run_experiment(config, models, seeds, data_dir, output_root):
    cmd = [
        sys.executable,
        "scripts/run_experiment.py",
        "--config", config,
        "--models", *models,
        "--seeds", *[str(s) for s in seeds],
        "--data-dir", str(data_dir),
        "--output-root", str(output_root),
    ]

    print("\nCOMMAND:")
    print(" ".join(cmd))

    subprocess.run(
        cmd,
        cwd=REPO_DIR,
        check=True,
    )

def is_complete_run(run_dir):
    required = [
        run_dir / "metrics.json",
        run_dir / "predictions.csv",
        run_dir / "history.csv",
        run_dir / "coverage_stratified_metrics.csv",
    ]

    return all(
        p.exists() and p.stat().st_size > 0
        for p in required
    )

def completed_runs_count(root):
    if not root.exists():
        return 0

    return sum(
        1 for metrics in root.rglob("metrics.json")
        if is_complete_run(metrics.parent)
    )

def write_resume_manifest():
    rows = []

    for dataset_root in [
        RESULT_ROOT / "dailydialog_confirmatory_5seed",
        RESULT_ROOT / "meld_confirmatory_5seed",
    ]:
        if not dataset_root.exists():
            continue

        for metrics in sorted(dataset_root.rglob("metrics.json")):
            run_dir = metrics.parent

            if not is_complete_run(run_dir):
                continue

            m = json.loads(
                metrics.read_text(encoding="utf-8")
            )

            rows.append({
                "dataset": m.get("dataset"),
                "model": m.get("model"),
                "seed": m.get("seed"),
                "accuracy": m.get("accuracy"),
                "macro_f1": m.get("macro_f1"),
                "run_dir": str(
                    run_dir.relative_to(RESULT_ROOT)
                ),
            })

    manifest = WORK_ROOT / "resume_manifest.csv"

    with manifest.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        fields = [
            "dataset",
            "model",
            "seed",
            "accuracy",
            "macro_f1",
            "run_dir",
        ]

        writer = csv.DictWriter(
            f,
            fieldnames=fields,
        )

        writer.writeheader()
        writer.writerows(rows)

    return manifest, rows

def create_latest_snapshot():
    manifest, rows = write_resume_manifest()

    snapshot = WORK_ROOT / "ijmlc_confirmatory_latest.zip"

    if snapshot.exists():
        snapshot.unlink()

    with zipfile.ZipFile(
        snapshot,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as z:

        z.write(
            manifest,
            arcname="resume_manifest.csv",
        )

        for p in RESULT_ROOT.rglob("*"):
            if not p.is_file():
                continue

            rel = p.relative_to(RESULT_ROOT)

            if "feature_cache" in rel.parts:
                continue

            if "_tmp_checkpoints" in rel.parts:
                continue

            z.write(
                p,
                arcname=str(
                    Path("results") / "english" / rel
                ),
            )

    print(
        "\nSNAPSHOT UPDATED:",
        snapshot,
        "| complete runs:",
        len(rows),
        "/50",
        "| size:",
        round(snapshot.stat().st_size / 1024**2, 2),
        "MB",
    )

DD_OUT = RESULT_ROOT / "dailydialog_confirmatory_5seed"
ME_OUT = RESULT_ROOT / "meld_confirmatory_5seed"

for seed in FULL_SEEDS:

    print("\n" + "#" * 90)
    print(f"CONFIRMATORY SEED {seed}")
    print("#" * 90)

    run_experiment(
        "configs/dailydialog_roberta.yaml",
        CONFIRMATORY_MODELS,
        [seed],
        DD_DIR,
        DD_OUT,
    )

    run_experiment(
        "configs/meld_roberta.yaml",
        CONFIRMATORY_MODELS,
        [seed],
        MELD_DIR,
        ME_OUT,
    )

    print(
        "\nCompleted:",
        completed_runs_count(DD_OUT),
        "/25 DailyDialog |",
        completed_runs_count(ME_OUT),
        "/25 MELD",
    )

    create_latest_snapshot()

print("\nALL REQUESTED SEEDS FINISHED.")

In [ ]:
# ============================================================
# 12. VERIFY RESULTS
# ============================================================

DD_OUT = RESULT_ROOT / "dailydialog_confirmatory_5seed"
ME_OUT = RESULT_ROOT / "meld_confirmatory_5seed"

dd_done = completed_runs_count(DD_OUT)
me_done = completed_runs_count(ME_OUT)

print("DailyDialog complete:", dd_done, "/25")
print("MELD complete       :", me_done, "/25")

if dd_done == 25 and me_done == 25:
    print("\n50/50 CONFIRMATORY RUNS COMPLETE.")
else:
    print(
        "\nExperiment incomplete. "
        "Download ijmlc_confirmatory_latest.zip and resume later."
    )

for label, root in [
    ("DailyDialog", DD_OUT),
    ("MELD", ME_OUT),
]:
    summary = root / "summary.csv"

    if summary.exists():
        print("\n", label)
        display(pd.read_csv(summary))

In [ ]:
# ============================================================
# 13. PACKAGE FINAL RESULTS
# ============================================================

create_latest_snapshot()

final_zip = (
    WORK_ROOT
    / "ijmlc_english_confirmatory_5seed_results_final.zip"
)

if final_zip.exists():
    final_zip.unlink()

with zipfile.ZipFile(
    final_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as z:

    manifest = WORK_ROOT / "resume_manifest.csv"

    if manifest.exists():
        z.write(
            manifest,
            arcname="resume_manifest.csv",
        )

    for p in RESULT_ROOT.rglob("*"):
        if not p.is_file():
            continue

        rel = p.relative_to(RESULT_ROOT)

        if "_tmp_checkpoints" in rel.parts:
            continue

        z.write(
            p,
            arcname=str(
                Path("results") / "english" / rel
            ),
        )

    for rel in [
        "configs/dailydialog_roberta.yaml",
        "configs/meld_roberta.yaml",
        "ontology/ekman_appraisal_ontology_v2.rdf",
        "ontology/english_nrc_ontology_v2_build_summary.json",
        "scripts/run_experiment.py",
        "scripts/build_english_lexicon.py",
    ]:
        p = REPO_DIR / rel

        if p.exists():
            z.write(
                p,
                arcname=str(
                    Path("frozen_protocol") / rel
                ),
            )

print("\nFINAL ZIP:")
print(final_zip)
print(
    "Size:",
    round(final_zip.stat().st_size / 1024**2, 2),
    "MB",
)

print("\nDOWNLOAD AND SEND THIS FILE:")
print(
    "/kaggle/working/"
    "ijmlc_english_confirmatory_5seed_results_final.zip"
)

## Files to keep

If the run stops before 50/50:
- download `/kaggle/working/ijmlc_confirmatory_latest.zip`

If the run reaches 50/50:
- download `/kaggle/working/ijmlc_english_confirmatory_5seed_results_final.zip`

For a later resume, upload `ijmlc_confirmatory_latest.zip` as a private Kaggle dataset and attach it together with package v3 and the NRC dataset.